# Import packages

In [1]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os
import rasterio

# %matplotlib qt

## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

In [2]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# SAM 2.1 checkpoints. Download from:
# https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Set your directories

In [5]:
dir = os.chdir("C:/Users/leoni/kirgis_repo/segmenteverygrain/leonieexamples/2509_entireflight_nomask_2500promtlimit")
dir = os.getcwd()
os.makedirs("ouputgpkg", exist_ok=True)
os.makedirs("inputtiles", exist_ok=True)
os.makedirs("plots", exist_ok=True)
os.makedirs("csv_stats", exist_ok=True)

inputtiledir = os.path.join(dir, "inputtiles/")
ouputgpkg = os.path.join(dir, "ouputgpkg/")
csvdir = os.path.join(dir, "csv_stats")
plotdir = os.path.join(dir, "plots/")

## Inlcuding masks

In [9]:
# ==========================================================
# MASKEN-SEKTION (ab hier)
# 1 = NICHT segmentieren
# 0 oder nodata = segmentieren erlaubt
# ==========================================================
veg_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\vegetation\veg_masks_rgbvi_fallback\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_vegmask_rgbvi.tif"
shadow_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\shadow\0-02\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_shadowmask.tif"

import glob
from rasterio.warp import reproject, Resampling

def read_mask_for_tile(tile_path, full_mask_src, mask_band=1):
    """
    Schneidet/projiziert die Vollmaske auf das Grid des Tiles.
    Ergebnis: 2D-Array mit exakt gleicher Höhe/Breite wie das Tile.
    """
    with rasterio.open(tile_path) as tsrc:
        tile_mask = np.zeros((tsrc.height, tsrc.width), dtype=np.float32)

        reproject(
            source=rasterio.band(full_mask_src, mask_band),
            destination=tile_mask,
            src_transform=full_mask_src.transform,
            src_crs=full_mask_src.crs,
            dst_transform=tsrc.transform,
            dst_crs=tsrc.crs,
            resampling=Resampling.nearest,   # wichtig für binäre Masken
            dst_nodata=np.nan
        )

    return tile_mask


# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")

if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")


# --- loop (mit Masken + Shape-Fix + RGB bleibt UNVERÄNDERT) ---
# Ziel: segmentieren NUR wo Maske == 0 oder NoData
#       NICHT segmentieren wo Maske == 1
# WICHTIG: RGB wird NICHT auf 0 gesetzt (weil 0 ein gültiger RGB-Wert sein kann).
#          Stattdessen: image_pred maskieren + Prompts filtern + mask_all final hart beschneiden.

with rasterio.open(veg_mask_path) as veg_src, rasterio.open(shadow_mask_path) as shadow_src:

    for i, fname in enumerate(tiles, start=1):
        tile_id = os.path.splitext(os.path.basename(fname))[0]
        print(f"[{i}/{len(tiles)}] {tile_id}")

        # --------------------------------------------------
        # 1) load RGB tile (unverändert)
        # --------------------------------------------------
        image = np.array(load_img(fname))  # (H, W, 3) typischerweise

        # --------------------------------------------------
        # 2) tile-aligned masks aus Vollmasken holen
        # --------------------------------------------------
        veg_tile = read_mask_for_tile(fname, veg_src, mask_band=1)
        shadow_tile = read_mask_for_tile(fname, shadow_src, mask_band=1)

        # Deine Logik:
        # 1 = IGNORE (nicht segmentieren)
        # 0 oder NoData = segmentieren erlaubt
        #
        # (robust gegen 1/255 etc.: alles >0 wird als "maskiert" interpretiert)
        veg_exclude = np.isfinite(veg_tile) & (veg_tile > 0)
        shadow_exclude = np.isfinite(shadow_tile) & (shadow_tile > 0)

        exclude_mask = veg_exclude | shadow_exclude
        valid_mask = ~exclude_mask  # True = hier darf segmentiert werden

        # --------------------------------------------------
        # 3) SHAPE CHECK / FIX (TIFF: load_img kann H/W anders liefern)
        # --------------------------------------------------
        img_hw = image.shape[:2]      # (H, W)
        mask_hw = valid_mask.shape    # (H, W) erwartet

        if mask_hw != img_hw:
            if mask_hw[::-1] == img_hw:
                valid_mask = valid_mask.T
                print(f"Transposed valid_mask from {mask_hw} to {valid_mask.shape} to match image {img_hw}")
            else:
                raise ValueError(
                    f"Shape mismatch between image and valid_mask: "
                    f"image.shape[:2]={img_hw}, valid_mask.shape={mask_hw}"
                )

        if not np.any(valid_mask):
            print("Tile fully masked by vegetation/shadow -> skipping")
            continue

        # --------------------------------------------------
        # 4) UNet prediction
        # --------------------------------------------------
        image_pred = seg.predict_image(image, unet, I=256)

        # Falls prediction-Shape nicht passt -> valid_mask ggf. transponieren
        pred_hw = image_pred.shape[:2]
        if pred_hw != valid_mask.shape:
            if valid_mask.shape[::-1] == pred_hw:
                valid_mask = valid_mask.T
                print(f"Transposed valid_mask to match image_pred: {pred_hw}")
            else:
                raise ValueError(
                    f"Shape mismatch between image_pred and valid_mask: "
                    f"image_pred.shape[:2]={pred_hw}, valid_mask.shape={valid_mask.shape}"
                )

        # --------------------------------------------------
        # 5) Maskierung VOR Segmentierung:
        #    -> Nur image_pred maskieren (Prompt-Quelle)
        #    -> RGB NICHT verändern!
        # --------------------------------------------------
        image_pred = np.array(image_pred, copy=True)
        image_pred[~valid_mask] = 0

        image_for_seg = image  # unverändert lassen

        # --------------------------------------------------
        # 6) Prompts (Unet -> points)
        # --------------------------------------------------
        labels, coords = seg.label_grains(image_for_seg, image_pred, dbs_max_dist=10.0)

        # --------------------------------------------------
        # 7) Prompts außerhalb valid_mask entfernen (Sicherheitsfilter)
        #    Achtung: labels kann 2D Label-Bild sein -> nicht immer filtern!
        # --------------------------------------------------
        coords = np.asarray(coords)

        if len(coords) > 0:
            # Annahme: coords = (x, y)
            x_idx = np.clip(np.round(coords[:, 0]).astype(int), 0, valid_mask.shape[1] - 1)
            y_idx = np.clip(np.round(coords[:, 1]).astype(int), 0, valid_mask.shape[0] - 1)

            keep = valid_mask[y_idx, x_idx]
            coords = coords[keep]

            # labels nur filtern, wenn es wirklich ein 1D Prompt-Labelarray ist
            try:
                labels_arr = np.asarray(labels)
                if labels_arr.ndim == 1 and len(labels_arr) == len(keep):
                    labels = labels_arr[keep]
            except Exception:
                pass

        if len(coords) == 0:
            print("No valid prompts after masking -> skipping tile")
            continue

        # --------------------------------------------------
        # 8) SAM segmentation (arbeitet mit originalem RGB + maskierter Prediction)
        # --------------------------------------------------
        all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
            sam,
            image_for_seg,
            image_pred,
            coords,
            labels,
            min_area=min_area_px,
            plot_image=True,
            remove_edge_grains=True,
            remove_large_objects=True,
        )

        # --------------------------------------------------
        # 9) Finaler Hard-Cut:
        #    Garantiert: in maskierten Bereichen gibt es keine Segmente
        # --------------------------------------------------
        try:
            mask_all = np.array(mask_all, copy=True)
            if mask_all.shape != valid_mask.shape:
                if mask_all.shape[::-1] == valid_mask.shape:
                    mask_all = mask_all.T
                else:
                    print(f"Warning: mask_all shape {mask_all.shape} != valid_mask shape {valid_mask.shape}")

            mask_all[~valid_mask] = 0
        except Exception as e:
            print(f"Could not apply final hard mask cut: {e}")

        # --------------------------------------------------
        # 10) Plot speichern
        # --------------------------------------------------
        figname = os.path.join(plotdir, f"{tile_id}.png")
        fig.savefig(figname, dpi=200, bbox_inches="tight")
        plt.close(fig)

        print("Saved the Plot")
        print("done with segmentation")

Found 1 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/1] tile_r000016_c000012
segmenting image tiles...


100%|██████████| 3/3 [00:01<00:00,  2.66it/s]


creating masks using SAM...


  1%|          | 66/9427 [00:04<11:11, 13.94it/s]


KeyboardInterrupt: 

## Not including Masks and just saving the pngs

## Run segmentation

Collect tiles

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

# --- optional: prompt cap for SAM (None = no limit) ---
MAX_SAM_PROMPTS = 2500
PROMPT_SUBSAMPLE_MODE = "random"   # "random" oder "first"
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")


if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")



# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

        # --- optional: cap number of prompts before SAM ---
    coords = np.asarray(coords)
    if MAX_SAM_PROMPTS is not None and len(coords) > MAX_SAM_PROMPTS:
        n_before = len(coords)
    
        if PROMPT_SUBSAMPLE_MODE == "first":
            keep_idx = np.arange(MAX_SAM_PROMPTS)
        else:  # random
            keep_idx = np.sort(rng.choice(len(coords), size=MAX_SAM_PROMPTS, replace=False))
    
        coords = coords[keep_idx]
    
        # labels nur mitfiltern, wenn es ein 1D prompt-label array ist
        try:
            labels_arr = np.asarray(labels)
            if labels_arr.ndim == 1 and len(labels_arr) == n_before:
                labels = labels_arr[keep_idx]
        except Exception:
            pass
    
        print(f"Prompt cap active: reduced prompts from {n_before} -> {len(coords)}")
    
    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=min_area_px,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )
    figname =os.path.join(plotdir, f"{tile_id}.png")

    fig.savefig(figname, dpi = 200, bbox_inches="tight")
    plt.close(fig)
    print("Saved the Plot")
    print("done with segmentation")






Found 1 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/1] tile_r000017_c000012
segmenting image tiles...


100%|██████████| 3/3 [00:01<00:00,  2.53it/s]


creating masks using SAM...


100%|██████████| 301/301 [00:21<00:00, 14.13it/s]


finding overlapping polygons...


...

finding best polygons...


100%|██████████| 99/99 [00:00<00:00, 115.84it/s]


creating labeled image...


100%|██████████| 103/103 [00:00<00:00, 231.34it/s]


Saved the Plot
done with segmentation


## Not including Masks and saving all statistiks, plots and images and gpkgs

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

# zusätzliche Imports für Nachbearbeitung (ohne deine Library-Zelle zu ändern)
import geopandas as gpd
from shapely.geometry import Polygon
from skimage.measure import regionprops_table

# --- optional: prompt cap for SAM (None = no limit) ---
MAX_SAM_PROMPTS = 2500
PROMPT_SUBSAMPLE_MODE = "random"   # "random" oder "first"
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")

if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")


# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

    # --- optional: cap number of prompts before SAM ---
    coords = np.asarray(coords)
    if MAX_SAM_PROMPTS is not None and len(coords) > MAX_SAM_PROMPTS:
        n_before = len(coords)

        if PROMPT_SUBSAMPLE_MODE == "first":
            keep_idx = np.arange(MAX_SAM_PROMPTS)
        else:  # random
            keep_idx = np.sort(rng.choice(len(coords), size=MAX_SAM_PROMPTS, replace=False))

        coords = coords[keep_idx]

        # labels nur mitfiltern, wenn es ein 1D prompt-label array ist
        try:
            labels_arr = np.asarray(labels)
            if labels_arr.ndim == 1 and len(labels_arr) == n_before:
                labels = labels_arr[keep_idx]
        except Exception:
            pass

        print(f"Prompt cap active: reduced prompts from {n_before} -> {len(coords)}")

    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=min_area_px,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )

    # --------------------------------------------------
    # 1) Segmentierungsplot speichern (dein bestehender Plot)
    # --------------------------------------------------
    seg_plot_path = os.path.join(plotdir, f"{tile_id}.png")
    fig.savefig(seg_plot_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved segmentation plot")

    # --------------------------------------------------
    # 2) Polygone georeferenzieren (Pixel -> UTM / CRS des Tiles)
    # --------------------------------------------------
    with rasterio.open(fname) as dataset:
        projected_polys = []
        for grain in all_grains:
            # grain.exterior.xy liefert (x_coords, y_coords) in Pixelkoordinaten
            # hier: row = y, col = x
            px_x = np.asarray(grain.exterior.xy[0])
            px_y = np.asarray(grain.exterior.xy[1])

            x_proj, y_proj = rasterio.transform.xy(
                dataset.transform,
                px_y,   # rows
                px_x    # cols
            )

            poly = Polygon(np.vstack((x_proj, y_proj)).T)
            projected_polys.append(poly)

        # GeoDataFrame erzeugen
        gdf = gpd.GeoDataFrame({"geometry": projected_polys}, geometry="geometry", crs=dataset.crs)

        # --------------------------------------------------
        # 3) Eigenschaften / Statistiken aus labeled image
        # --------------------------------------------------
        # (intensity_image hier weggelassen, da nur geometrische Properties gebraucht werden)
        props = regionprops_table(
            labels.astype("int"),
            properties=("label", "area", "centroid", "major_axis_length", "minor_axis_length"),
        )
        grain_stats = pd.DataFrame(props)

        # Falls aus irgendeinem Grund Anzahl nicht exakt passt, robust behandeln
        if len(grain_stats) != len(gdf):
            nmin = min(len(grain_stats), len(gdf))
            print(f"Warning: len(gdf)={len(gdf)} != len(grain_stats)={len(grain_stats)} -> truncating to {nmin}")
            gdf = gdf.iloc[:nmin].copy()
            grain_stats = grain_stats.iloc[:nmin].copy()

        # Zentroiden (row,col) -> CRS coords
        centroid_x, centroid_y = rasterio.transform.xy(
            dataset.transform,
            grain_stats["centroid-0"].values,  # rows
            grain_stats["centroid-1"].values,  # cols
        )

        # Pixelgröße aus Transform (X-Richtung); für quadratische Pixel ok
        px_size_x = abs(dataset.transform.a)

        # Attribute in GDF schreiben
        gdf["label"] = grain_stats["label"].values
        gdf["area_px"] = grain_stats["area"].values
        gdf["centroid_x"] = centroid_x
        gdf["centroid_y"] = centroid_y
        gdf["major_axis_px"] = grain_stats["major_axis_length"].values
        gdf["minor_axis_px"] = grain_stats["minor_axis_length"].values
        gdf["major_axis_length"] = grain_stats["major_axis_length"].values * px_size_x
        gdf["minor_axis_length"] = grain_stats["minor_axis_length"].values * px_size_x
        gdf["major_axis_mm"] = gdf["major_axis_length"] * 1000.0
        gdf["minor_axis_mm"] = gdf["minor_axis_length"] * 1000.0

    # --------------------------------------------------
    # 4) Histogramm-Plot speichern (NICHT den Segmentierungsplot überschreiben)
    # --------------------------------------------------
    if len(gdf) > 0:
        fig_hist, ax_hist = seg.plot_histogram_of_axis_lengths(
            gdf["major_axis_mm"],
            gdf["minor_axis_mm"],
            binsize=0.25,
            xlimits=[8, 2 * 256],
        )
        hist_plot_path = os.path.join(plotdir, f"{tile_id}_hist.png")  # eigener Name!
        fig_hist.savefig(hist_plot_path, dpi=200, bbox_inches="tight")
        plt.close(fig_hist)
        print("Saved histogram plot")
    else:
        print("No grains found -> skipping histogram plot")

    # --------------------------------------------------
    # 5) GPKG + Statistik-CSV speichern
    # --------------------------------------------------
    gpkg_path = os.path.join(ouputgpkg, f"{tile_id}_grains.gpkg")
    csv_path = os.path.join(csvdir, f"{tile_id}_grain_stats.csv")

    # GPKG
    gdf.to_file(gpkg_path, driver="GPKG")
    print(f"Saved GPKG: {gpkg_path}")

    # CSV (ohne Geometrie; Geometrie steckt im GPKG)
    stats_df = gdf.drop(columns="geometry").copy()
    stats_df.to_csv(csv_path, index=False)
    print(f"Saved stats CSV: {csv_path}")

    print("done with segmentation + export")

Found 740 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/740] tile_r000004_c000004
segmenting image tiles...


100%|██████████| 3/3 [00:01<00:00,  2.65it/s]


creating masks using SAM...


0it [00:00, ?it/s]


finding overlapping polygons...


...

163it [00:00, 244.95it/s]


finding overlapping polygons...


162it [00:00, 243.17it/s]


finding best polygons...


 19%|█▉        | 6/31 [00:00<00:01, 17.69it/s]